# Matching total and reduced magnetic scalar potentials

This independently specified linear magnetostatic principle case has a planar material interface, relative permeabilities 100 and 1, and a quadratic harmonic potential. It demonstrates explicit nonzero reduced boundary values and the CPU matching-trace route. SI units are used throughout.

This introduction does **not** certify all mixed Omega formulations: overall formulation parity remains **OPEN**. Exact polynomial reproduction here applies to this geometry, source, polynomial order, and declared evaluation stencil. It is not an interface theorem, a performance benchmark, or a nonlinear/MMM validation. Kelvin validation is a separate executable sample: `validation_test/magnetostatics/mixed_omega_kelvin_coupled_probe.py`.

Notebook viewing additionally requires `anywidget` (the NGSolve WebGUI viewer); this is not a CPU solver dependency.

In [1]:
import json
import anywidget  # explicit notebook viewer dependency
import numpy as np
import ngsolve as ng
import radia
from netgen.occ import Box, Glue, OCCGeometry, Pnt, Z
from ngsolve.webgui import Draw
from radia.kelvin_solver import solve_magnetostatic_matching_trace_total_reduced_omega

ng.SetNumThreads(1)
order = 2
mu_r = {"total": 100.0, "reduced": 1.0}
print({"radia": radia.__version__, "ngsolve": ng.__version__, "backend": "cpu", "order": order})

{'radia': '5.3.3', 'ngsolve': '6.2.2608', 'backend': 'cpu', 'order': 2}


The lower region stores a total potential. The upper region stores a reduced potential: physical field reconstruction includes the declared source field. Consequently its prescribed boundary value is the physical potential minus the source potential. Setting that boundary value to zero would describe a different problem.

In [2]:
total = Box(Pnt(-1,-1,-1), Pnt(1,1,0)); total.mat("total")
total.faces.name = "outer_total"; total.faces.Max(Z).name = "interface"
reduced = Box(Pnt(-1,-1,0), Pnt(1,1,1)); reduced.mat("reduced")
reduced.faces.name = "outer_reduced"; reduced.faces.Min(Z).name = "interface"
geometry = Glue([total, reduced])
mesh = ng.Mesh(OCCGeometry(geometry).GenerateMesh(maxh=0.65))
Draw(mesh, mesh, name="material_mesh", order=2, show=True)


WebGLScene

In [3]:
tangent = -ng.x - 0.4*ng.y + 0.02*(ng.x**2-ng.y**2)
normal = 1 + 0.05*ng.x + 0.02*ng.y
phi_total = tangent - ng.z*normal/100
phi_air = tangent - ng.z*normal
phi_source = -ng.x - 0.4*ng.y - ng.z + 0.011*(ng.x**2-ng.y**2)
axes = (ng.x, ng.y, ng.z)
def minus_gradient(phi):
    return -ng.CF(tuple(phi.Diff(axis) for axis in axes))
source_H = minus_gradient(phi_source)
exact_H = mesh.MaterialCF({"total": minus_gradient(phi_total), "reduced": minus_gradient(phi_air)})
space = ng.H1(mesh, order=order)
trace, total_boundary, reduced_boundary = [ng.GridFunction(space) for _ in range(3)]
with ng.TaskManager():
    for field, expression in [(trace, phi_source), (total_boundary, phi_total),
                              (reduced_boundary, phi_air-phi_source)]:
        field.Set(expression, bonus_intorder=6)
    result = solve_magnetostatic_matching_trace_total_reduced_omega(
        mesh, source_H, trace, order=order, solver="cg", cg_preconditioner="local",
        cg_tolerance=1e-13, bonus_intorder=6, mu_r_by_material=mu_r,
        reduced_materials=("reduced",), total_materials=("total",),
        interface_boundary="interface", surface_dirichlet={
            "total": {"outer_total": total_boundary},
            "reduced": {"outer_reduced": reduced_boundary}})
    error2 = ng.Integrate(ng.InnerProduct(result["H_cf"]-exact_H,
                                         result["H_cf"]-exact_H), mesh, order=8)
    reference2 = ng.Integrate(ng.InnerProduct(exact_H, exact_H), mesh, order=8)
relative_rms = float(np.sqrt(error2/reference2))
assert result["linear_residual_relative"] <= 1e-6
assert relative_rms < 1e-8
print(json.dumps({"relative_H_rms": relative_rms,
                  "true_relative_residual": result["linear_residual_relative"],
                  "elements": mesh.ne, "dofs": result["fes"].ndof,
                  "formulation_parity_status": "OPEN"}, indent=2))

{
  "relative_H_rms": 1.3273172684229222e-13,
  "true_relative_residual": 5.263833637719858e-14,
  "elements": 266,
  "dofs": 503,
  "formulation_parity_status": "OPEN"
}


In [4]:
# Check physical normal B on a declared nine-point interior interface stencil.
points = [(x,y) for x in (-0.6,0.0,0.6) for y in (-0.6,0.0,0.6)]
epsilon = 1e-8
lower = np.array([result["B_cf"](mesh(x,y,-epsilon))[2] for x,y in points])
upper = np.array([result["B_cf"](mesh(x,y,epsilon))[2] for x,y in points])
relative_normal_jump = float(np.linalg.norm(upper-lower)/np.linalg.norm((upper+lower)/2))
nonzero_reduced_value = float(reduced_boundary(mesh(0.3,0.2,1.0)))
assert relative_normal_jump < 1e-8
assert abs(nonzero_reduced_value) > 1e-4
print(json.dumps({"interface_samples": len(points),
                  "relative_normal_B_jump": relative_normal_jump,
                  "prescribed_reduced_boundary_sample": nonzero_reduced_value}, indent=2))

{
  "interface_samples": 9,
  "relative_normal_B_jump": 1.2043981202378816e-14,
  "prescribed_reduced_boundary_sample": -0.018549999999998554
}


In [5]:
Draw(result["H_cf"], mesh, name="H_A_per_m", order=2,
     draw_surf=False, draw_vol=True, autoscale=True, show=True)

WebGLScene

The vector-field RMS compares the Euclidean difference of physical H fields. The true residual certifies the assembled linear solve; it is a separate condition from field accuracy. The saved scenes show the mesh and primary field. Production applications use MCP or Simulink, not notebook workbench controls.